In [2]:
#%pip install "ecdsa>=0.18.0"
from __future__ import annotations

import hashlib
import json
import time
from typing import Any, Optional

from ecdsa import BadSignatureError, SECP256k1, SigningKey, VerifyingKey

print("ecdsa ready — curve SECP256k1")

ecdsa ready — curve SECP256k1


In [3]:
#Question 1

sk_demo = SigningKey.generate(curve=SECP256k1)
vk_demo = sk_demo.get_verifying_key()

print("Private key", sk_demo.to_string().hex())
print("Public key", vk_demo.to_string().hex())
print("Public key length (bytes):  ", len(vk_demo.to_string()))
print("Private key length (bytes):  ", len(sk_demo.to_string()))


Private key 9e1f56d7718ab611b8ae99830f60420b7d45d32b743524aa80bc2cd87dbfc94b
Public key ad1bde212a84329a66cde28daaf2d1fe9d376491b7697d48673ad6c7a10aedd49f43be0983ebcc1d1822f66cf7bb80eb2fa61c1369469b487c31e0619f38f372
Public key length (bytes):   64
Private key length (bytes):   32
Address: 7454e9056d2e90d110945024a75497d8aee6d305
Length:  40 (expect 40 hex chars)


In [4]:
class Wallet:
    """ECDSA wallet wrapper (curve = SECP256k1)."""

    def __init__(self, signing_key: SigningKey, label: str = "") -> None:
        self.label = label
        self._sk = signing_key
        self._vk = signing_key.get_verifying_key()

    @classmethod
    def create(cls, label: str = "") -> "Wallet":
        """Generate a fresh SECP256k1 key pair."""
        return cls(SigningKey.generate(curve=SECP256k1), label=label)

    @property
    def public_key_hex(self) -> str:
        return self._vk.to_string().hex()

    @property
    def private_key_hex(self) -> str:
        """For local demos only — never log or commit in production paths."""
        return self._sk.to_string().hex()

    @property
    def address(self) -> str:
        return pubkey_to_address(self._vk.to_string())

    def sign(self, message: bytes) -> str:
        """Return ECDSA signature as hex."""
        return self._sk.sign(message).hex()

    @staticmethod
    def verify(message: bytes, signature_hex: str, public_key_hex: str) -> bool:
        """Return True iff signature is valid for message under public_key_hex."""
        vk = VerifyingKey.from_string(bytes.fromhex(public_key_hex), curve=SECP256k1)
        try:
            return vk.verify(bytes.fromhex(signature_hex), message)
        except BadSignatureError:
            return False


print("Wallet class ready.")

Wallet class ready.


In [7]:
alice = Wallet.create("Alice")
bob = Wallet.create("Bob")

print(f"{alice.label} address:", alice.address)
print(f"{bob.label} address:  ", bob.address)
print("Distinct?", alice.address != bob.address)
print("Alice pubkey (short):", alice.public_key_hex[:16], "...")
print("Bob pubkey (short):  ", bob.public_key_hex[:16], "...")


Alice address: 87d9aa239eb4ed285859512980175392f0a4f93a
Bob address:   51b66e6d5527efc093dde87d1d46be9bd367e15e
Distinct? True
Alice pubkey (short): c41fae5886f32c90 ...
Bob pubkey (short):   25d48df1db9bca4b ...


In [11]:
#Question 2
def pubkey_to_address(pubkey_bytes: bytes) -> str:
    """Simplified pedagogical address: SHA256(pubkey).hexdigest()[:40]."""
    return hashlib.sha256(pubkey_bytes).hexdigest()[:40]


demo_address = pubkey_to_address(vk_demo.to_string())
print("Address:", demo_address)
print("Length: ", len(demo_address), "(expect 40 hex chars)")
assert len(demo_address) == 40
assert all(c in "0123456789abcdef" for c in demo_address)

Address: 7454e9056d2e90d110945024a75497d8aee6d305
Length:  40 (expect 40 hex chars)


In [8]:
#Question 3 

def canonical_tx_payload(
    sender: str,
    recipient: str,
    amount: float,
    timestamp: float,
) -> bytes:
    """Canonical bytes to sign: JSON of fields EXCLUDING signature."""
    body = {
        "amount": amount,
        "recipient": recipient,
        "sender": sender,
        "timestamp": timestamp,
    }
    return json.dumps(body, sort_keys=True, separators=(",", ":")).encode("utf-8")


# Fixed timestamp so the demo is reproducible within this session
TS = 1_500_000_000.0
AMOUNT = 10.0

msg = canonical_tx_payload(alice.address, bob.address, AMOUNT, TS)
print("Canonical payload:")
print(msg.decode("utf-8"))
print("Payload length (bytes):", len(msg))

Canonical payload:
{"amount":10.0,"recipient":"51b66e6d5527efc093dde87d1d46be9bd367e15e","sender":"87d9aa239eb4ed285859512980175392f0a4f93a","timestamp":1500000000.0}
Payload length (bytes): 147


In [9]:
signature_hex = alice.sign(msg)

ok_original = Wallet.verify(msg, signature_hex, alice.public_key_hex)
ok_wrong_key = Wallet.verify(msg, signature_hex, bob.public_key_hex)

print("Signature (hex, truncated):", signature_hex[:32], "...")
print("verify original (Alice key):", ok_original)
print("verify with Bob's key:      ", ok_wrong_key)

assert ok_original is True
assert ok_wrong_key is False

Signature (hex, truncated): 4f670bb4bd175ecd529e00bc7dfe0694 ...
verify original (Alice key): True
verify with Bob's key:       False


In [10]:
mutated = canonical_tx_payload(alice.address, bob.address, 1000.0, TS)

ok_mutated = Wallet.verify(mutated, signature_hex, alice.public_key_hex)

print("Original payload:", msg.decode("utf-8"))
print("Mutated payload: ", mutated.decode("utf-8"))
print()
print("verify original:", Wallet.verify(msg, signature_hex, alice.public_key_hex))
print("verify mutated: ", ok_mutated)

assert ok_mutated is False
print("\nFailure OK: changing the amount invalidates the signature.")

Original payload: {"amount":10.0,"recipient":"51b66e6d5527efc093dde87d1d46be9bd367e15e","sender":"87d9aa239eb4ed285859512980175392f0a4f93a","timestamp":1500000000.0}
Mutated payload:  {"amount":1000.0,"recipient":"51b66e6d5527efc093dde87d1d46be9bd367e15e","sender":"87d9aa239eb4ed285859512980175392f0a4f93a","timestamp":1500000000.0}

verify original: True
verify mutated:  False

Failure OK: changing the amount invalidates the signature.


In [ ]:
#Question 4